# Prompt tuning on PubMedQA

## Setup

### Imports

In [1]:
import json
import random

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter
from peft import PromptTuningConfig, PromptTuningInit, get_peft_model

/Users/andrew/Documents/Berkeley/MIDS/DS266/Final Project/DS266_RAG/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1115 00:13:11.207000 85958 .venv/lib/python3.14/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


### Settings

In [2]:
# model to fine-tune
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# run on mac gpu if possible
if torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("using:", device)

using: mps


### Load data

In [3]:
# load 1,000 pubmedqa questions
with open("pubmedqa/data/ori_pqal.json") as f:
    data = json.load(f)

questions = []
abstracts = []
answers = []
long_answers = []

for entry in data.values():
    questions.append(entry["QUESTION"])
    abstracts.append(" ".join(entry["CONTEXTS"]))
    answers.append(entry["final_decision"])
    long_answers.append(entry["LONG_ANSWER"])

print(len(questions))

1000


### Split training and test data

In [4]:
# split first 700 for training, last 300 for testing
train_questions = questions[:700]
train_abstracts = abstracts[:700]
train_answers = answers[:700]

test_questions = questions[700:]
test_abstracts = abstracts[700:]
test_answers = answers[700:]
test_long_answers = long_answers[700:]

print("train:", len(train_questions))
print("test:", len(test_questions))

train: 700
test: 300


## Load model

In [5]:
# load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float32)
model = model.to(device)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 320.60it/s]


## Prompt and answer

In [6]:
# build the message we send the model for one question
def build_prompt(question, abstract):
    instructions = "Read the abstract and answer the question. Answer with only one word: yes, no, or maybe."
    question_text = f"Abstract: {abstract}\n\nQuestion: {question}"

    conversation = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": question_text},
    ]

    # put the conversation in qwen's chat format
    prompt = tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
    return prompt

In [16]:
# the three answers the model can pick from
labels = ["yes", "no", "maybe"]

# get the token number for each answer
label_token_ids = []
for label in labels:
    token_id = tokenizer.encode(label, add_special_tokens=False)[0]
    label_token_ids.append(token_id)

print(label_token_ids)


# have the model pick yes, no, or maybe for one question
def pick_answer(which_model, question, abstract):
    prompt = build_prompt(question, abstract)
    prompt_tokens = tokenizer(prompt, return_tensors="pt").to(device)

    # run the model once, no training here
    with torch.no_grad():
        output = which_model(**prompt_tokens)

    # the model's score for every possible next word
    next_word_scores = output.logits[0, -1]

    # grab the scores for just our three answers
    label_scores = []
    for token_id in label_token_ids:
        label_scores.append(next_word_scores[token_id].item())

    # keep the answer with the highest score
    best_index = label_scores.index(max(label_scores))
    return labels[best_index]

[9693, 2152, 36760]


In [17]:
# try it on one test question
print("correct:", test_answers[0])
print("model:", pick_answer(model, test_questions[0], test_abstracts[0]))

correct: yes
model: maybe


## Score before training

In [9]:
# ask the model every test question
predictions_before = []
for i in range(len(test_questions)):
    question = test_questions[i]
    abstract = test_abstracts[i]
    predictions_before.append(pick_answer(model, question, abstract))

In [10]:
print("accuracy:", accuracy_score(test_answers, predictions_before))
print("macro f1:", f1_score(test_answers, predictions_before, average="macro"))

accuracy: 0.11666666666666667
macro f1: 0.06965174129353234


In [11]:
# count how often the model gave each answer
print("model:", Counter(predictions_before))
print("correct:", Counter(test_answers))

model: Counter({'maybe': 300})
correct: Counter({'yes': 155, 'no': 110, 'maybe': 35})


## Set up prompt tuning

In [12]:
# prompt tuning settings
prompt_config = PromptTuningConfig(
    task_type="CAUSAL_LM",
    num_virtual_tokens=20,
    prompt_tuning_init=PromptTuningInit.TEXT,
    prompt_tuning_init_text="Read the abstract and answer the question with yes, no, or maybe.",
    tokenizer_name_or_path=model_name,
)

# freeze the model and add the soft prompt we'll train
prompt_model = get_peft_model(model, prompt_config)
prompt_model.print_trainable_parameters()

trainable params: 17,920 || all params: 494,050,688 || trainable%: 0.0036


In [13]:
# how wrong the model is on one training question
def get_loss(question, abstract, answer):
    prompt = build_prompt(question, abstract)
    prompt_tokens = tokenizer(prompt, return_tensors="pt").to(device)

    # run the model with the soft prompt added
    output = prompt_model(**prompt_tokens)
    next_word_scores = output.logits[0, -1]

    # token number of the correct answer
    correct_token_id = label_token_ids[labels.index(answer)]
    correct_token = torch.tensor(correct_token_id).to(device)

    # big when the model gives the right answer a low score
    loss = torch.nn.functional.cross_entropy(next_word_scores, correct_token)
    return loss


# try it on one training question
print(get_loss(train_questions[0], train_abstracts[0], train_answers[0]))

tensor(4.9053, device='mps:0', grad_fn=<NllLossBackward0>)


In [14]:
# same random shuffle every run
random.seed(42)

# updates only the soft prompt, the rest of the model is frozen
optimizer = torch.optim.AdamW(prompt_model.parameters(), lr=3e-2)

## Train

In [15]:
# train on all 700 training questions once, in random order
prompt_model.train()

order = list(range(len(train_questions)))
random.shuffle(order)

total_loss = 0
for step in range(len(order)):
    i = order[step]
    loss = get_loss(train_questions[i], train_abstracts[i], train_answers[i])

    # figure out which way to nudge the soft prompt, then nudge it
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    total_loss += loss.item()

    # check progress every 100 questions
    if (step + 1) % 100 == 0:
        print(step + 1, "average loss:", total_loss / 100)
        total_loss = 0

100 average loss: 2.389858640953898
200 average loss: 1.0967293624021113
300 average loss: 1.0228016126155852
400 average loss: 1.0414057394862175
500 average loss: 0.9132322588562966
600 average loss: 0.8798612605035305
700 average loss: 0.9323250617831945


## Score after training

In [18]:
# switch from training mode to answering mode
prompt_model.eval()

# ask the trained model every test question
predictions_after = []
for i in range(len(test_questions)):
    question = test_questions[i]
    abstract = test_abstracts[i]
    predictions_after.append(pick_answer(prompt_model, question, abstract))

print("accuracy:", accuracy_score(test_answers, predictions_after))
print("macro f1:", f1_score(test_answers, predictions_after, average="macro"))

accuracy: 0.33666666666666667
macro f1: 0.31800691553509747


In [19]:
# count how often the trained model gave each answer
print("model:", Counter(predictions_after))
print("correct:", Counter(test_answers))

model: Counter({'maybe': 206, 'yes': 78, 'no': 16})
correct: Counter({'yes': 155, 'no': 110, 'maybe': 35})
